In [1]:
import os
import openeo

In [2]:
backend_url = os.getenv("OPENEO_BACKEND_URL")
connection = openeo.connect(backend_url).authenticate_oidc()

Authenticated using refresh token.


## Showcase of query process alone

The query can be used on its own as a STAC catalogue interface. It is called here to show the result returned to the client or the next process.
The result is just a FeatureCollection with the items found by the query:

In [3]:
collection_url = "https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c"
force_query_pg = connection.datacube_from_process(
    "query_stac",
    url=collection_url,
    temporal_extent=["2026-04-08", "2026-04-09"],
    spatial_extent={ "west": -15.6, "south": 15.1, "east": -15.4, "north": 15.4}
)

res = connection.execute(force_query_pg)
res

{'features': [{'assets': {'B01': {'alternate': {'https': {'alternate:name': 'HTTPS',
       'auth:refs': ['oidc'],
       'href': 'https://download.dataspace.copernicus.eu/odata/v1/Products(7f36be26-b273-461c-abf0-15f1cdf448e0)/Nodes(S2C_MSIL1C_20260408T112111_N0512_R037_T28PDC_20260408T134338.SAFE)/Nodes(GRANULE)/Nodes(L1C_T28PDC_A008301_20260408T113407)/Nodes(IMG_DATA)/Nodes(T28PDC_20260408T112111_B01.jp2)/$value',
       'storage:refs': []}},
     'alternate:name': 'S3',
     'auth:refs': ['s3'],
     'bands': [{'description': 'Coastal aerosol (band 1)',
       'eo:center_wavelength': 0.443,
       'eo:common_name': 'coastal',
       'eo:full_width_half_max': 0.228,
       'name': 'B01'}],
     'data_type': 'uint16',
     'file:checksum': '16205925a84c28946b3662bee7a0e48348a588d7957c4bc586c5184718941c16067f',
     'file:local_path': 'S2C_MSIL1C_20260408T112111_N0512_R037_T28PDC_20260408T134338.SAFE/GRANULE/L1C_T28PDC_A008301_20260408T113407/IMG_DATA/T28PDC_20260408T112111_B01.jp2',


### Enclosure links

For further processing in force, we use the "enclosure" link type, which points to the entire SAFE archive for download by the EOAP. This is in principle independent of the query process, but shows how it is used our case:

In [4]:
[l for l in res["features"][0]["links"] if l["rel"] == "enclosure"]

[{'href': 's3://eodata/Sentinel-2/MSI/L1C/2026/04/08/S2C_MSIL1C_20260408T112111_N0512_R037_T28PDC_20260408T134338.SAFE/',
  'rel': 'enclosure',
  'title': 'S3 path to source directory',
  'type': 'application/x-directory'}]

### Query process with FORCE EOAP

On our testing environment, it is not possible to run CWL based workflows. Therefore we use a dummy `run_udf` process which showcases how the query results can be processed by the EOAP. The query results are passed as context to the UDF and would be passed as one of the CWL parameters when using `run_cwl_to_stac` with FORCE or the FORCE custom process. Here, the parameter is named "inputs".

We use a dummy data cube `dummy_cube` so that we can call our udf with the `apply_datacube` signature. The `dummy_cube` has no relevance at all, only the `context` parameter is relevant.
The UDF itself also does not do anything, but note the code inside: In FORCE, the actual download would happen at this stage. The code included in the UDF is a sketch of how results may be downloaded inside the CWL workflow.

The structure is:

```Python
 item_collection = pystac.ItemCollection.from_dict(context["inputs"])
    for item in item_collection:
        # download
        # process
        # ...
```

In [5]:
dummy_cube = connection.load_collection("SENTINEL3_SYN_L2_SYN", temporal_extent=["2022-06-01", "2022-06-02"],  spatial_extent={ "west": -15.6, "south": 15.1, "east": -15.4, "north": 15.4})
#dummy_cube.download("test.nc")

In [6]:
UDF_CODE = """
from openeo.udf.debug import inspect
import pystac
import xarray

def apply_datacube(cube: xarray.DataArray, context: dict) -> xarray.DataArray:
    '''does nothing, but showcases how the query_stac derived context may be used'''

    item_collection = pystac.ItemCollection.from_dict(context["inputs"])
    for item in item_collection:
        enclosure_url = _extract_enclosure_link_url(item)
        if enclosure_url:
            inspect(data=enclosure_url, message=f"downloading {enclosure_url}")
        else:
            inspect(message=f"did not find enclosure for item {item}", level="warning")
   
    return cube

def _extract_enclosure_link_url(item: pystac.Item) -> str:
    enclosure_links = set(l.href for l in item.links if l.rel == "enclosure")
    try:
        url = enclosure_links.pop()
        inspect(data=url, message="found URL")
    except KeyError:
        inspect(data=[l.rel for l in item.links], message=f"Found rel types but no 'enclosure' for item '{item}'")
        url = None
    return url
    

"""
udf = openeo.UDF(UDF_CODE, context={"from_parameter": "context"})

In [7]:
dummy_result = dummy_cube.apply_dimension(
    udf,
    dimension="t",
    context={
        "inputs": force_query_pg,
    }
)

In [8]:
print(dummy_result.to_json())

{
  "process_graph": {
    "querystac1": {
      "process_id": "query_stac",
      "arguments": {
        "spatial_extent": {
          "west": -15.6,
          "south": 15.1,
          "east": -15.4,
          "north": 15.4
        },
        "temporal_extent": [
          "2026-04-08",
          "2026-04-09"
        ],
        "url": "https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c"
      }
    },
    "loadcollection1": {
      "process_id": "load_collection",
      "arguments": {
        "id": "SENTINEL3_SYN_L2_SYN",
        "spatial_extent": {
          "west": -15.6,
          "south": 15.1,
          "east": -15.4,
          "north": 15.4
        },
        "temporal_extent": [
          "2022-06-01",
          "2022-06-02"
        ]
      }
    },
    "applydimension1": {
      "process_id": "apply_dimension",
      "arguments": {
        "context": {
          "inputs": {
            "from_node": "querystac1"
          }
        },
        "data": {
         

In [9]:
dummy_result.download("dummy_result.nc")